In [60]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit Learn
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score, mean_squared_error, log_loss, mean_absolute_error

# Audio
import librosa
import librosa.display

# MiniLearn — your from-scratch library
# from minilearn.classifiers import LogisticRegression, KNN, GaussianNaiveBayes, DecisionTreeClassifier
# from minilearn.preprocessing import StandardScaler, train_test_split
# from minilearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [56]:
# load processed data
metadata = pd.read_csv("/home/westley/Documents/classes/spring_2026/machine_learning/Project_CSE432-532/SER_Project/processed_data/ravdess_metadata.csv")
features = pd.read_csv("/home/westley/Documents/classes/spring_2026/machine_learning/Project_CSE432-532/SER_Project/processed_data/ravdess_features.csv")

# load dataframes
df = metadata.merge(features, on="filename", how="inner")
features = features.drop(columns=["filename"])

In [ ]:
# Predicting emotional intensity from features

X = features.copy()
y = df["intensity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.mean(axis=0)[:5]
X_train_scaled.std(axis=0)[:5]

# Baseline model

classifier = LogisticRegression(max_iter=2000)
classifier.fit(X_train_scaled, y_train)

y_pred = classifier.predict(X_test_scaled)
y_proba = classifier.predict_proba(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average="macro"))
print("Recall:", recall_score(y_test, y_pred, average="macro"))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Log Loss", log_loss(y_test, y_proba))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7515274949083504
Precision: 0.756438557763061
Recall: 0.7442430917100521
Macro F1: 0.7455915330252787
Log Loss 0.5159395773569019
[[222  42]
 [ 80 147]]


This isn't bad. We see it at least is better than random selection.

In [61]:
# Predicting emotional intensity from features

X = features.copy()
y = df["intensity_code"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))

MAE: 0.4123985630750532
MSE: 0.3501377340692657
RMSE: 0.5917243733946285
R²: -0.40854951053184907


These are pretty bad. Especially R^2 which shows that the model is worse at predicting intensity than simply guessing the mean intensity every time. And the average prediction error is about 59%.